# WorldSim DGX Spark QLoRA Smoke Run

Thin Jupyter wrapper around the shared `training.lib.qlora_smoke` API. Use this notebook to verify the DGX Spark kernel, run `preflight` / `smoke` / `longer_smoke`, and inspect post-run JSON structure without duplicating trainer logic.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_marker = cwd / "training/lib/qlora_smoke.py"
assert repo_marker.exists(), f"Run this notebook from the repo root. cwd={cwd}"
{
    "python_executable": sys.executable,
    "cwd": str(cwd),
    "repo_marker": str(repo_marker),
}


{'python_executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3',
 'cwd': '/home/hyunlord/github/worldsim-training',
 'repo_marker': '/home/hyunlord/github/worldsim-training/training/lib/qlora_smoke.py'}

## Run Mode

Edit only this cell to switch between `preflight`, `smoke`, and `longer_smoke`. The shared helper resolves the final step/sample counts so the notebook stays thin.


In [2]:
from training.lib.qlora_smoke import resolve_notebook_run_mode

RUN_MODE = "longer_smoke"  # preflight | smoke | longer_smoke
RUN_ID = None  # set an explicit string to reuse a fixed output folder

resolved_run = resolve_notebook_run_mode(RUN_MODE, run_id=RUN_ID)
RUN_MODE = resolved_run["run_mode"]
RUN_ID = resolved_run["run_id"]
MAX_STEPS = resolved_run["max_steps"]
MAX_TRAIN_SAMPLES = resolved_run["max_train_samples"]
MAX_EVAL_SAMPLES = resolved_run["max_eval_samples"]

resolved_run


{'max_steps': 25,
 'max_train_samples': 256,
 'max_eval_samples': 64,
 'run_mode': 'longer_smoke',
 'run_id': '20260311T165040Z'}

## Environment Visibility

Run this before training. If CUDA or bitsandbytes looks wrong here, stop and switch to the correct DGX Spark kernel.


In [3]:
from training.lib.qlora_smoke import get_environment_summary

environment = get_environment_summary()
torch_info = environment.get("torch", {})
bnb_info = environment.get("bitsandbytes", {})
{
    "python_executable": environment.get("python", {}).get("executable"),
    "cwd": environment.get("cwd"),
    "torch_version": torch_info.get("version"),
    "torch_cuda_version": torch_info.get("cuda_version"),
    "torch_cuda_available": torch_info.get("cuda_available"),
    "gpu_count": torch_info.get("cuda_device_count", 0),
    "gpu_names": torch_info.get("cuda_device_names", []),
    "bitsandbytes_available": bnb_info.get("available"),
    "bitsandbytes_version": bnb_info.get("version"),
    "environment_summary": environment,
}


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python_executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3',
 'cwd': '/home/hyunlord/github/worldsim-training',
 'torch_version': '2.12.0.dev20260225+cu130',
 'torch_cuda_version': '13.0',
 'torch_cuda_available': True,
 'gpu_count': 1,
 'gpu_names': ['NVIDIA GB10'],
 'bitsandbytes_available': True,
 'bitsandbytes_version': '0.49.2',
 'environment_summary': {'python': {'version': '3.12.3',
   'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'},
  'platform': {'system': 'Linux',
   'release': '6.14.0-1015-nvidia',
   'machine': 'aarch64'},
  'cwd': '/home/hyunlord/github/worldsim-training',
  'torch': {'available': True,
   'version': '2.12.0.dev20260225+cu130',
   'cuda_version': '13.0',
   'cuda_available': True,
   'mps_available': False,
   'cuda_device_count': 1,
   'cuda_device_name': 'NVIDIA GB10',
   'cuda_device_names': ['NVIDIA GB10'],
   'cuda_bf16_supported': True},
  'transformers': {'available': True, 'version': '5.2.0'},
  'dataset

## Strict True-QLoRA Preflight

This uses the same shared runtime detection as the CLI path and hard-fails before training if true CUDA 4-bit QLoRA cannot be satisfied.


In [4]:
from training.lib.qlora_smoke import get_true_qlora_preflight

preflight = get_true_qlora_preflight()
assert preflight["ok"], preflight["blocker_reason"]
preflight


{'ok': True,
 'runtime': {'device': 'cuda',
  'use_qlora': True,
  'fallback_reason': None,
  'torch_dtype': 'bfloat16'},
 'environment': {'python': {'version': '3.12.3',
   'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'},
  'platform': {'system': 'Linux',
   'release': '6.14.0-1015-nvidia',
   'machine': 'aarch64'},
  'cwd': '/home/hyunlord/github/worldsim-training',
  'torch': {'available': True,
   'version': '2.12.0.dev20260225+cu130',
   'cuda_version': '13.0',
   'cuda_available': True,
   'mps_available': False,
   'cuda_device_count': 1,
   'cuda_device_name': 'NVIDIA GB10',
   'cuda_device_names': ['NVIDIA GB10'],
   'cuda_bf16_supported': True},
  'transformers': {'available': True, 'version': '5.2.0'},
  'datasets': {'available': True, 'version': '4.7.0'},
  'peft': {'available': True, 'version': '0.18.1'},
  'accelerate': {'available': True, 'version': '1.12.0'},
  'bitsandbytes': {'available': True, 'version': '0.49.2'}},
 'blocker_reason': None}

## Config

This cell builds `SmokeRunConfig` from the resolved run mode. Keep edits here if you need to override model or batch sizing.


In [5]:
from pathlib import Path

from training.lib.qlora_smoke import DEFAULT_MODEL_NAME, SmokeRunConfig

OUTPUT_DIR = Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1") / RUN_ID

config = SmokeRunConfig(
    model_name=DEFAULT_MODEL_NAME,
    train_file=Path("data/training/worldsim-v31-mix-v1/train_converted.jsonl"),
    dev_file=Path("data/training/worldsim-v31-mix-v1/dev_converted.jsonl"),
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_eval_samples=MAX_EVAL_SAMPLES,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    require_qlora=True,
    seed=42,
)

config.to_dict()


{'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'train_file': 'data/training/worldsim-v31-mix-v1/train_converted.jsonl',
 'dev_file': 'data/training/worldsim-v31-mix-v1/dev_converted.jsonl',
 'output_dir': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z',
 'max_steps': 25,
 'max_train_samples': 256,
 'max_eval_samples': 64,
 'max_length': 512,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 1,
 'learning_rate': 0.0002,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'seed': 42,
 'trust_remote_code': False,
 'disable_qlora': False,
 'require_qlora': True,
 'dry_run': False}

## Run Smoke Training

This cell stays intentionally thin and calls the shared API directly.


In [6]:
import time

from training.lib.qlora_smoke import run_smoke_or_raise

started_at = time.perf_counter()
result = run_smoke_or_raise(config)
elapsed_seconds = round(time.perf_counter() - started_at, 2)
{
    "elapsed_seconds": elapsed_seconds,
    "result": result.to_dict(),
}


Loading weights:   1%|          | 3/290 [00:01<03:40,  1.30it/s, Materializing param=model.layers.0.mlp.down_proj.weight]/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 290/290 [00:04<00:00, 58.53it/s, Materializing param=model.norm.weight]                               
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1258: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passe

Step,Training Loss
1,3.805990
2,3.225099
3,2.990781
4,2.593345
5,2.823926
6,2.531184
7,2.328074
8,2.333749
9,2.047080
10,1.931737


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'elapsed_seconds': 67.43,
 'result': {'success': True,
  'status': 'ok',
  'used_true_qlora': True,
  'runtime': {'device': 'cuda',
   'use_qlora': True,
   'fallback_reason': None,
   'torch_dtype': 'bfloat16'},
  'environment': {'python': {'version': '3.12.3',
    'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'},
   'platform': {'system': 'Linux',
    'release': '6.14.0-1015-nvidia',
    'machine': 'aarch64'},
   'cwd': '/home/hyunlord/github/worldsim-training',
   'torch': {'available': True,
    'version': '2.12.0.dev20260225+cu130',
    'cuda_version': '13.0',
    'cuda_available': True,
    'mps_available': False,
    'cuda_device_count': 1,
    'cuda_device_name': 'NVIDIA GB10',
    'cuda_device_names': ['NVIDIA GB10'],
    'cuda_bf16_supported': True},
   'transformers': {'available': True, 'version': '5.2.0'},
   'datasets': {'available': True, 'version': '4.7.0'},
   'peft': {'available': True, 'version': '0.18.1'},
   'accelerate': {'available': True

## Inspect Run Artifacts

These cells group the key JSON artifacts so you can quickly see what configuration ran, whether true QLoRA stayed active, and what the train/eval metrics look like.


In [7]:
from training.lib.qlora_smoke import load_json_artifact, preview_metrics

run_config = load_json_artifact(result.output_dir, "run_config.json")
summary = load_json_artifact(result.output_dir, "summary.json")
metrics = preview_metrics(result.output_dir)

{
    "run_config": {
        "model_name": run_config.get("config", {}).get("model_name"),
        "max_steps": run_config.get("config", {}).get("max_steps"),
        "max_train_samples": run_config.get("config", {}).get("max_train_samples"),
        "max_eval_samples": run_config.get("config", {}).get("max_eval_samples"),
        "require_qlora": run_config.get("config", {}).get("require_qlora"),
        "output_dir": run_config.get("output_dir"),
    },
    "summary": {
        "status": summary.get("status"),
        "used_true_qlora": summary.get("used_true_qlora"),
        "train_loss": summary.get("train_loss"),
        "eval_loss": summary.get("eval_loss"),
        "finite_losses": summary.get("finite_losses"),
        "train_rows": summary.get("train_rows"),
        "eval_rows": summary.get("eval_rows"),
    },
    "metrics": {
        "train_runtime": metrics.get("train_runtime"),
        "train_steps_per_second": metrics.get("train_steps_per_second"),
        "train_loss": metrics.get("train_loss"),
        "eval_loss": metrics.get("eval_loss"),
        "eval_runtime": metrics.get("eval_runtime"),
    },
}


{'run_config': {'model_name': None,
  'max_steps': None,
  'max_train_samples': None,
  'max_eval_samples': None,
  'require_qlora': None,
  'output_dir': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z'},
 'summary': {'status': 'ok',
  'used_true_qlora': True,
  'train_loss': 1.901280493736267,
  'eval_loss': 1.1026809215545654,
  'finite_losses': True,
  'train_rows': 256,
  'eval_rows': 64},
 'metrics': {'train_runtime': None,
  'train_steps_per_second': None,
  'train_loss': None,
  'eval_loss': None,
  'eval_runtime': None}}

## Sample Generation Structure Inspection

Use the shared sample-generation helpers to separate raw parse failures, recoverable fenced JSON, truly malformed JSON, and enum drift.


In [8]:
from training.lib.qlora_smoke import load_sample_generations, summarize_sample_generations
from tools.generation_analyzer import generate_report

samples = load_sample_generations(result.output_dir)
sample_summary = summarize_sample_generations(samples)
analysis_report = generate_report(samples, examples_per_category=3)

{
    "raw_parseable_json": sample_summary.get("raw_parseable_json"),
    "fenced_json": sample_summary.get("fenced_json"),
    "fence_stripped_parseable_json": sample_summary.get("fence_stripped_parseable_json"),
    "recoverable_fenced_json": sample_summary.get("recoverable_fenced_json"),
    "malformed_json": sample_summary.get("malformed_json"),
    "enum_drift_total": sample_summary.get("enum_drift_total"),
    "semantic_valid": sample_summary.get("semantic_valid"),
    "semantic_low_quality": sample_summary.get("semantic_low_quality"),
    "semantic_drift": sample_summary.get("semantic_drift"),
    "language_drift": sample_summary.get("language_drift"),
    "classifications": sample_summary.get("classifications"),
    "failure_categories": sample_summary.get("failure_categories"),
    "analyzer_counts": analysis_report.get("counts_by_failure_category"),
    "analyzer_overall_status": analysis_report.get("overall_status"),
}


{'raw_parseable_json': 7,
 'fenced_json': 0,
 'fence_stripped_parseable_json': 7,
 'recoverable_fenced_json': 0,
 'malformed_json': 0,
 'enum_drift_total': 4,
 'semantic_valid': 0,
 'semantic_low_quality': 1,
 'semantic_drift': 0,
 'language_drift': 0,
 'classifications': {'enum_drift': 3, 'raw_parseable': 4},
 'failure_categories': {'enum_drift': 3, 'ok': 4},
 'analyzer_counts': {'enum_drift': 1, 'ok': 4, 'prompt_leakage': 2},
 'analyzer_overall_status': 'prompt_leakage_issue'}

In [9]:
analysis_report.get("example_failures", {})


{'enum_drift': [{'task': 'B',
   'generated_assistant': '{"text_ko": "과제", "text_en": "task", "register": "haera", "emotion_expressed": ["joy", "sadness", "fear", "anger", "trust", "disgust", "surprise", "anticipation"], "temperament_influence": ["melancholic", "melancholic_rhyme", "melancholic_sense", "melancholic_mood", "melancholic_tone", "melancholic_vocalization", "melancholic_phoneme", "melancholic_accent"], "stress_level": 0.4}',
   'json_failure': {'raw_parseable_json': True,
    'fence_stripped_parseable_json': True,
    'fenced_json': False,
    'raw_json_parse_error': None,
    'stripped_json_parse_error': None,
    'category': 'ok'},
   'prompt_leakage': None,
   'enum_drift': [{'field_name': 'emotion_expressed',
     'expected_enum_set': ['anger',
      'anticipation',
      'disgust',
      'fear',
      'joy',
      'sadness',
      'surprise',
      'trust'],
     'actual_value': '["joy", "sadness", "fear", "anger", "trust", "disgust", "surprise", "anticipation"]',
    

In [10]:
samples[:3]


[{'task': 'A',
  'source_split': 'dev',
  'prompt_messages': [{'role': 'system',
    'content': '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라. `*_ko` 필드는 순우리말로, `*_en` 필드는 자연스러운 영어로 쓰고, 길이와 문장 수를 정확히 지켜라.'},
   {'role': 'user',
    'content': '[과제] 아래 인물의 기질과 성격을 이중언어 JSON으로만 묘사하라.\n\n[TEMP]\nNS=0.7 HA=0.3 RD=0.8 P=0.3 type=sanguine\n\n[기질 이름]\n다혈질\n\n[기질 키워드]\n낙천적, 사교적, 산만함, 열정적\n\n[성격 이름]\n신중한원로\n\n[성격 설명]\n위험을 경계하고 규율을 중시하는 어른\n\n[성격 키워드]\n겁많음, 꼼꼼함, 조용함, 규율중시\n\n[STRESS]\n0.3\n\n[WORLD]\ndefault\n\n[WORLD_DESC]\n석기시대, 지상자원 정상, 일반 생태계\n\n[WORLD_VOCAB]\n\n\n[어투 코드]\nhaera\n\n[변형번호]\n1\n\n[생성 규칙]\n- JSON object 하나만 출력하라.\n- markdown fence를 쓰지 마라.\n- 설명문을 덧붙이지 마라.\n- 첫 글자는 반드시 { 여야 한다.\n- 형식 예시나 placeholder 문구를 복사하지 마라.\n- 길이 설명문, enum 설명문, schema 설명문을 값으로 쓰지 마라.\n- 각 field에는 실제 값만 채워라.\n- 모든 key 이름과 문자열 값은 JSON 큰따옴표를 써라.\n- key 순서는 text_ko, text_en, register, dominant_trait, temperament_expressed 이다.\n- 모든 문자열 값은 JSON 큰따옴표를 지켜라.\n- text_ko와 text_en에는 실제

## Final Operational Judgment

This is the compact status dict to use for go/no-go decisions after a smoke run.


In [11]:
from training.lib.qlora_smoke import build_operational_judgment

final_status = build_operational_judgment(summary, sample_summary, output_dir=result.output_dir)
{
    "final_status": final_status,
    "analyzer_overall_status": analysis_report.get("overall_status"),
    "example_failure_categories": sorted(analysis_report.get("example_failures", {}).keys()),
    "semantic_summary": {
        "semantic_valid": sample_summary.get("semantic_valid"),
        "semantic_low_quality": sample_summary.get("semantic_low_quality"),
        "semantic_drift": sample_summary.get("semantic_drift"),
        "language_drift": sample_summary.get("language_drift"),
    },
    "recommended_next_action": final_status.get("recommended_next_action"),
}


{'final_status': {'true_qlora_passed': True,
  'raw_json_parse_failed': False,
  'raw_parseable_json': 7,
  'fence_stripped_parseable_json': 7,
  'recoverable_fenced_json': 0,
  'malformed_json': 0,
  'enum_drift_total': 4,
  'operational_issue': 'enum_drift_present',
  'recommended_next_action': 'Investigate post-train structural quality and enum compliance before broader smoke conclusions.',
  'output_dir': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z'},
 'analyzer_overall_status': 'prompt_leakage_issue',
 'example_failure_categories': ['enum_drift', 'prompt_leakage'],
 'semantic_summary': {'semantic_valid': 0,
  'semantic_low_quality': 1,
  'semantic_drift': 0,
  'language_drift': 0},
 'recommended_next_action': 'Investigate post-train structural quality and enum compliance before broader smoke conclusions.'}

## Next Recommended Config

If the current run is stable, use this next conservative config for the next escalation step.


In [12]:
from training.lib.qlora_smoke import recommended_next_smoke_config

next_run = recommended_next_smoke_config()
{
    "current_run_mode": RUN_MODE,
    "current_output_dir": str(OUTPUT_DIR),
    "suggested_next_config": next_run,
}


{'current_run_mode': 'longer_smoke',
 'current_output_dir': 'outputs/smoke_cuda_notebook/worldsim-v31-mix-v1/20260311T165040Z',
 'suggested_next_config': {'max_steps': 50,
  'max_train_samples': 512,
  'max_eval_samples': 128,
  'per_device_train_batch_size': 1,
  'per_device_eval_batch_size': 1,
  'gradient_accumulation_steps': 1}}